In [26]:
import json
import os
import shutil

In [27]:
img_dir = "E:/Research/simplify-me/simplify_me_dataset/images"
meta_path = "E:/Research/simplify-me/simplify_me_dataset/meta.json"

save_dir = "E:/Research/simplify-me-dataset/"

In [28]:
def load_data():
    with open(meta_path, 'r', encoding='utf-8') as f:
        return json.load(f)

In [29]:
def generate_unique_id(benchmark, img_id):
    return f'{benchmark}-{img_id}'

In [30]:
def get_and_preprocess_data(data, split, ds_size):
    processed_data = []

    data = data[split]

    data = sorted(
        data,
        key=lambda x: float(x['dpo']['sle_delta']),
        reverse=True
    )

    if ds_size != -1:
        data = data[:ds_size]

    print(f"split: {split}, data lenngth: {len(data)}, min sle delta: {data[-1]['dpo']['sle_delta']}")

    benchmark_count = {}

    for item in data:

        if item['benchmark'] not in benchmark_count:
            benchmark_count[item['benchmark']] = 0

            os.makedirs(os.path.join(save_dir, split, 'images', item['benchmark']), exist_ok=True)

        benchmark_count[item['benchmark']] += 1

        item['unique_id'] = generate_unique_id(item['benchmark'], item['id'])

        processed_data.append(item)

    print(benchmark_count)
    filename = f'annotations-{split}{'' if split != 'train' else '-full' if ds_size == -1 else '-' + str(ds_size)}.json'

    with open(os.path.join(save_dir, split, filename), 'w', encoding='utf-8') as fp:
        json.dump(processed_data, fp, indent=4, ensure_ascii=False)

    return filename

In [32]:
raw_data = load_data()
max_size = [-1, 10000, 5000, 1000]

for sz in max_size:
    fn = get_and_preprocess_data(raw_data, 'train', sz)
    fn = get_and_preprocess_data(raw_data, 'test', 5000)

split: train, data lenngth: 92174, min sle delta: 2.000012755393982
{'viswiz_train': 13451, 'flickr30k_val': 14933, 'coco_2014_train': 31122, 'viswiz_val': 4374, 'coco_2014_val': 15310, 'textcaps_train': 11302, 'textcaps_val': 1682}
split: test, data lenngth: 5000, min sle delta: 2.7845916748046875
{'viswiz_train': 669, 'flickr30k_val': 710, 'coco_2014_train': 985, 'coco_2014_val': 482, 'viswiz_val': 210, 'nocaps_test': 1402, 'textcaps_train': 467, 'textcaps_val': 75}
split: train, data lenngth: 10000, min sle delta: 3.366675615310669
{'viswiz_train': 1917, 'flickr30k_val': 2444, 'coco_2014_train': 2412, 'viswiz_val': 647, 'coco_2014_val': 1134, 'textcaps_train': 1227, 'textcaps_val': 219}
split: test, data lenngth: 5000, min sle delta: 2.7845916748046875
{'viswiz_train': 669, 'flickr30k_val': 710, 'coco_2014_train': 985, 'coco_2014_val': 482, 'viswiz_val': 210, 'nocaps_test': 1402, 'textcaps_train': 467, 'textcaps_val': 75}
split: train, data lenngth: 5000, min sle delta: 3.6479693055

## Test split

### Image copying

In [33]:
with open(os.path.join(save_dir, 'test', 'annotations-test.json'), 'r', encoding='utf-8') as test_fp:
    test_data = json.load(test_fp)

In [35]:
for data in test_data:
    raw_img_path = os.path.join(img_dir, data['benchmark'], data['image']['file_name'])
    target_img_path = os.path.join(save_dir, 'test', 'images', data['benchmark'], data['image']['file_name'])

    shutil.copy(raw_img_path, target_img_path)

### Generated captions

In [36]:
test_data_ids = {}

for data in test_data:
    test_data_ids[data['unique_id']] = data

In [37]:
generated_caption_folder = "E:/Research/gen-captions/gen-captions/scored"

In [38]:
experiments = {
    'gemma3-3-epoch-1000-trained.json': 'gemma3-trained-1000.json',
    'gemma3-3-epoch-10000-trained.json': 'gemma3-trained-10000.json',
    'gemma3-3-epoch-full-trained.json': 'gemma3-trained-full.json',
    'gemma3-base.json': 'gemma3-zeroshot.json',
    'Llama-3.2-11B-Vision-base.json': 'llama32-zeroshot.json',
    'Llama-32-11B-Vision-1000-trained.json': 'llama32-trained-1000.json',
    'Llama-32-11B-Vision-10000-trained.json': 'llama32-trained-10000.json',
    'Llama-32-11B-Vision-full-trained.json': 'llama32-trained-full.json',
    'Qwen2-VL-2B-Instruct-1000-trained.json': 'qwen2-2b-trained-1000.json',
    'Qwen2-VL-2B-Instruct-10000-trained.json': 'qwen2-2b-trained-10000.json',
    'Qwen2-VL-2B-Instruct-base.json': 'qwen2-2b-zeroshot.json',
    'Qwen2-VL-2B-Instruct-full-trained.json': 'qwen2-2b-trained-full.json',
    'Qwen2-VL-7B-Instruct-1000-trained.json': 'qwen2-7b-trained-1000.json',
    'Qwen2-VL-7B-Instruct-10000-trained.json': 'qwen2-7b-trained-10000.json',
    'Qwen2-VL-7B-Instruct-base.json': 'qwen2-7b-zeroshot.json',
    'Qwen2-VL-7B-Instruct-full-trained.json': 'qwen2-7b-trained-full.json',
    'Qwen25-VL-7B-Instruct-1000-trained.json': 'qwen25-7b-trained-1000.json',
    'Qwen25-VL-7B-Instruct-10000-trained.json': 'qwen25-7b-trained-10000.json',
    'Qwen25-VL-7B-Instruct-base.json': 'qwen25-7b-zeroshot.json',
    'Qwen25-VL-7B-Instruct-full-trained.json': 'qwen25-7b-trained-full.json'
}

In [39]:
generated_caption_files = os.listdir(generated_caption_folder)

for generated_caption_file in generated_caption_files:
    with open(os.path.join(generated_caption_folder, generated_caption_file), 'r', encoding='utf-8') as fp:
        generated_caption = json.load(fp)

    selected_generated_captions = []
    all_generated_captions = []

    for item in generated_caption:
        compiled_item = {
            'unique_id': generate_unique_id(item['benchmark'], item['id']),
            'id': item['id'],
            'benchmark': item['benchmark'],
            'file_name': item['images'][0].split('/')[-1],
            'instruction': item['instruction'],
            'generated_caption': item['model_output'],
            'sle_score': item['sle_score']['sle'][0],
        }

        all_generated_captions.append(compiled_item)

        if generate_unique_id(item['benchmark'], item['id']) in test_data_ids:
            selected_generated_captions.append(compiled_item)

    with open(os.path.join(save_dir, 'test', 'all_generated_captions', experiments[generated_caption_file]), 'w',
              encoding='utf-8') as fp:
        json.dump(all_generated_captions, fp, indent=5)

    if '10000' in generated_caption_file or 'base' in generated_caption_file:
        with open(os.path.join(save_dir, 'test', 'selected_generated_captions', experiments[generated_caption_file]),
                  'w', encoding='utf-8') as fp:
            json.dump(selected_generated_captions, fp, indent=5)

In [48]:
zeroshot_trained_map = [
    {
        'experiment': 'gemma3',
        'zero_shot': 'gemma3-zeroshot.json',
        'trained': 'gemma3-trained-1000.json'
    },
    {
        'experiment': 'llama3.2',
        'zero_shot': 'llama32-zeroshot.json',
        'trained': 'llama32-trained-1000.json'
    },
    {
        'experiment': 'qwen2-2b',
        'zero_shot': 'qwen2-2b-zeroshot.json',
        'trained': 'qwen2-2b-trained-1000.json'
    },
    {
        'experiment': 'qwen2-7b',
        'zero_shot': 'qwen2-7b-zeroshot.json',
        'trained': 'qwen2-7b-trained-1000.json'
    },
    {
        'experiment': 'qwen2.5-7b',
        'zero_shot': 'qwen25-7b-zeroshot.json',
        'trained': 'qwen25-7b-trained-1000.json'
    },
]

In [44]:
import numpy as np
from nltk import word_tokenize
import nltk
# nltk.download('punkt_tab')

for expt in zeroshot_trained_map:
    zeroshot_file = os.path.join(save_dir, 'test', 'selected_generated_captions', expt['zero_shot'])
    trained_file = os.path.join(save_dir, 'test', 'selected_generated_captions', expt['trained'])

    zeroshot_lens = []
    trained_lens = []

    with open(zeroshot_file, 'r', encoding='utf-8') as fp:
        zeroshot_data = json.load(fp)

    with open(trained_file, 'r', encoding='utf-8') as fp:
        trained_data = json.load(fp)


    for item in zeroshot_data:
        zeroshot_lens.append(len(word_tokenize(item['generated_caption'])))

    for item in trained_data:
        trained_lens.append(len(word_tokenize(item['generated_caption'])))

    print(f"{expt['experiment']}\tZeroshot: {np.mean(zeroshot_lens):.2f}\tTrained: {np.mean(trained_lens):.2f}. Ratio: {np.mean(trained_lens)/np.mean(zeroshot_lens):.2f}")

gemma3	Zeroshot: 90.57	Trained: 66.64. Ratio: 0.74
llama3.2	Zeroshot: 179.46	Trained: 42.75. Ratio: 0.24
qwen2-2b	Zeroshot: 39.22	Trained: 12.36. Ratio: 0.32
qwen2-7b	Zeroshot: 42.75	Trained: 8.37. Ratio: 0.20
qwen2.5-7b	Zeroshot: 71.67	Trained: 51.40. Ratio: 0.72


In [49]:
import numpy as np
from nltk import word_tokenize
import nltk
# nltk.download('punkt_tab')

for expt in zeroshot_trained_map:
    zeroshot_file = os.path.join(save_dir, 'test', 'all_generated_captions', expt['zero_shot'])
    trained_file = os.path.join(save_dir, 'test', 'all_generated_captions', expt['trained'])

    zeroshot_lens = []
    trained_lens = []


    with open(zeroshot_file, 'r', encoding='utf-8') as fp:
        zeroshot_data = json.load(fp)

    with open(trained_file, 'r', encoding='utf-8') as fp:
        trained_data = json.load(fp)

    for item in zeroshot_data:
        zeroshot_lens.append(len(word_tokenize(item['generated_caption'])))

    for item in trained_data:
        trained_lens.append(len(word_tokenize(item['generated_caption'])))

    print(f"{expt['experiment']}\tZeroshot: {np.mean(zeroshot_lens):.2f}\tTrained: {np.mean(trained_lens):.2f}. Ratio: {np.mean(trained_lens)/np.mean(zeroshot_lens):.2f}")

gemma3	Zeroshot: 89.66	Trained: 84.22. Ratio: 0.94
llama3.2	Zeroshot: 176.60	Trained: 120.47. Ratio: 0.68
qwen2-2b	Zeroshot: 36.67	Trained: 36.59. Ratio: 1.00
qwen2-7b	Zeroshot: 40.91	Trained: 37.82. Ratio: 0.92
qwen2.5-7b	Zeroshot: 70.58	Trained: 70.48. Ratio: 1.00
